In [ ]:
import sys
if 'google.colab' in sys.modules:
    !uv pip install --system --pre ngsolve

# The Function Space H(div)

$\newcommand{\opdiv}{\operatorname{div}}$
$\newcommand{\optrn}{\operatorname{tr}_n}$


A vector valued function space with well-defined divergence:

$$
H(\operatorname{div}) = \{ \sigma \in [L_2(\Omega)]^2 : \operatorname{div} \sigma \in L_2 \}
$$

Has continuous normal trace operator:

\begin{eqnarray*}
\optrn : H(\opdiv) & \rightarrow & H^{-1/2}(\partial \Omega) \\
\sigma & \rightarrow & \sigma \cdot n \quad \text{for smooth $\sigma$}
\end{eqnarray*}


Functions from $H(\opdiv)$ have single-valued normal components across sub-domain boundaries.


Brezzi-Douglas-Marini BDM finite elements:

$$
{\mathcal BDM}_k = \{ \sigma_h \in [P_k({\mathcal T})]^2 : [\sigma_h \cdot n] = 0 \}
$$

very similar: Raviart-Thomas RT elements

### Piola transformation:

mapping of domains:
$$
x = \Phi (\hat x)
$$

define Jacobian and determinant:

$$
F = \Phi^\prime, \; J = \det(F)
$$

Push-forward of vector fields by Piola transformation 

$$
{\mathcal P} \sigma (x)  := \frac{1}{J} F \hat \sigma(\hat x)
$$

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
import ipywidgets as widgets
from ipywidgets import interact, fixed

geo = MoveTo(0,-0.1).Rectangle(1,0.2).Face()
mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=0.05))

fes = HDiv(mesh, order=2)
# fes = VectorH1(mesh, order=2)

sigma = GridFunction(fes)
sigma.Set ( (2-100*y*y, 0))

In [ ]:
disp = GridFunction(VectorH1(mesh, order=2))
scene = Draw (sigma, vectors={"grid_size":30}) 

def myfunc(bend, thick): 
    disp.Set((-2*bend*x*y-0.5*y*y*(thick-1), bend*x*x + x*y*(thick-1)))
    mesh.SetDeformation(disp)
    scene.Redraw()
    mesh.UnsetDeformation()
    
    
params = widgets.interactive(myfunc,
        bend=widgets.FloatSlider(value=0, min=-0.2, max=0.2, step=0.001, 
                                                    description='bend:', disabled=False, 
                                                    continuous_update=True, orientation='horizontal',
                                                    readout=True, readout_format='.3f'),
        thick=widgets.FloatSlider(value=1, min=0.01, max=2, step=0.001, 
                                                    description='thick:', disabled=False, 
                                                    continuous_update=True, orientation='horizontal',
                                                    readout=True, readout_format='.3f'),)


display (params)


### Goal: 

* use $H(\operatorname{div})$ function space for approximation of flow velocities

* perfect for divergence free constraints

* discretization of viscosity term ? 
  - HDG methods
  - mixed stress methods 